# MagnaTagATune Dataset Exploration
## PhD Research: Amapiano-AI Audio Generation

This notebook explores the MagnaTagATune dataset as an interim training baseline for audio model fine-tuning.

In [ ]:
# Install required packages
!pip install -q librosa numpy pandas matplotlib seaborn torch torchaudio scikit-learn tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import torch
import torchaudio
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Download MagnaTagATune Dataset

The dataset is available at: https://mirg.city.ac.uk/codeapps/the-magnatagatune-dataset

In [ ]:
# Download MagnaTagATune dataset (if running on Colab)
# Note: The full dataset is ~200GB. For initial experiments, use a subset.

DATA_ROOT = Path('./magnatagatune')
DATA_ROOT.mkdir(exist_ok=True)

# Download annotations
!wget -q -nc -P {DATA_ROOT} https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv
!wget -q -nc -P {DATA_ROOT} https://mirg.city.ac.uk/datasets/magnatagatune/clip_info_final.csv

print("Annotations downloaded. For full audio, visit the dataset page.")

In [ ]:
# Load annotations
annotations_path = DATA_ROOT / 'annotations_final.csv'
clip_info_path = DATA_ROOT / 'clip_info_final.csv'

if annotations_path.exists():
    annotations = pd.read_csv(annotations_path, sep='\t')
    print(f"Loaded {len(annotations)} annotations")
    print(f"Number of tags: {len(annotations.columns) - 1}")
    display(annotations.head())
else:
    print("Please download the annotations file first.")

In [ ]:
# Analyze tag distribution
if 'annotations' in dir():
    tag_columns = [col for col in annotations.columns if col != 'clip_id' and col != 'mp3_path']
    tag_counts = annotations[tag_columns].sum().sort_values(ascending=False)
    
    # Plot top 50 tags
    plt.figure(figsize=(15, 8))
    tag_counts.head(50).plot(kind='bar')
    plt.title('Top 50 Tags in MagnaTagATune Dataset')
    plt.xlabel('Tag')
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTop 20 tags:\n{tag_counts.head(20)}")

## 2. Audio Feature Extraction

Extract features relevant for Amapiano music generation:
- Mel spectrograms
- MFCCs
- Chromagram
- Beat/tempo features

In [ ]:
class AudioFeatureExtractor:
    """Extract audio features for music analysis and generation."""
    
    def __init__(self, sr=22050, n_mels=128, n_mfcc=20, hop_length=512):
        self.sr = sr
        self.n_mels = n_mels
        self.n_mfcc = n_mfcc
        self.hop_length = hop_length
    
    def load_audio(self, file_path, duration=None):
        """Load audio file with optional duration limit."""
        y, sr = librosa.load(file_path, sr=self.sr, duration=duration)
        return y, sr
    
    def extract_mel_spectrogram(self, y):
        """Extract mel spectrogram."""
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_mels=self.n_mels, hop_length=self.hop_length
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        return mel_spec_db
    
    def extract_mfcc(self, y):
        """Extract MFCCs."""
        mfcc = librosa.feature.mfcc(
            y=y, sr=self.sr, n_mfcc=self.n_mfcc, hop_length=self.hop_length
        )
        return mfcc
    
    def extract_chroma(self, y):
        """Extract chromagram."""
        chroma = librosa.feature.chroma_stft(
            y=y, sr=self.sr, hop_length=self.hop_length
        )
        return chroma
    
    def extract_tempo_beats(self, y):
        """Extract tempo and beat positions."""
        tempo, beats = librosa.beat.beat_track(y=y, sr=self.sr)
        beat_times = librosa.frames_to_time(beats, sr=self.sr)
        return tempo, beat_times
    
    def extract_all_features(self, file_path, duration=None):
        """Extract all features from an audio file."""
        y, sr = self.load_audio(file_path, duration)
        
        features = {
            'mel_spectrogram': self.extract_mel_spectrogram(y),
            'mfcc': self.extract_mfcc(y),
            'chroma': self.extract_chroma(y),
            'tempo': self.extract_tempo_beats(y)[0],
            'duration': len(y) / sr
        }
        
        return features

# Initialize extractor
extractor = AudioFeatureExtractor()

In [ ]:
def visualize_audio_features(file_path, duration=30):
    """Visualize audio features for a single file."""
    features = extractor.extract_all_features(file_path, duration)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Mel spectrogram
    img1 = librosa.display.specshow(
        features['mel_spectrogram'], 
        x_axis='time', 
        y_axis='mel', 
        sr=extractor.sr,
        hop_length=extractor.hop_length,
        ax=axes[0]
    )
    axes[0].set_title('Mel Spectrogram')
    fig.colorbar(img1, ax=axes[0], format='%+2.0f dB')
    
    # MFCCs
    img2 = librosa.display.specshow(
        features['mfcc'],
        x_axis='time',
        ax=axes[1]
    )
    axes[1].set_title('MFCCs')
    fig.colorbar(img2, ax=axes[1])
    
    # Chromagram
    img3 = librosa.display.specshow(
        features['chroma'],
        x_axis='time',
        y_axis='chroma',
        ax=axes[2]
    )
    axes[2].set_title('Chromagram')
    fig.colorbar(img3, ax=axes[2])
    
    plt.suptitle(f"Tempo: {features['tempo']:.1f} BPM | Duration: {features['duration']:.1f}s")
    plt.tight_layout()
    plt.show()
    
    return features

# Example usage (uncomment when you have audio files)
# features = visualize_audio_features('path/to/audio.mp3')

## 3. Dataset Statistics

Analyze the dataset to understand its characteristics for Amapiano model training.

In [ ]:
# Amapiano-relevant tag analysis
amapiano_relevant_tags = [
    'drums', 'beat', 'electronic', 'dance', 'bass', 
    'synth', 'vocal', 'vocals', 'piano', 'house',
    'techno', 'ambient', 'rhythm', 'groove'
]

if 'annotations' in dir():
    relevant_tags_in_dataset = [tag for tag in amapiano_relevant_tags if tag in annotations.columns]
    print(f"Amapiano-relevant tags found in dataset: {relevant_tags_in_dataset}")
    
    if relevant_tags_in_dataset:
        relevant_counts = annotations[relevant_tags_in_dataset].sum().sort_values(ascending=False)
        
        plt.figure(figsize=(10, 6))
        relevant_counts.plot(kind='bar', color='coral')
        plt.title('Amapiano-Relevant Tags in MagnaTagATune')
        plt.xlabel('Tag')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 4. Data Preprocessing Pipeline

In [ ]:
class MagnaTagATuneDataset(torch.utils.data.Dataset):
    """PyTorch Dataset for MagnaTagATune."""
    
    def __init__(self, annotations_df, audio_dir, target_tags=None, 
                 sr=22050, duration=29.0, transform=None):
        self.annotations = annotations_df
        self.audio_dir = Path(audio_dir)
        self.sr = sr
        self.duration = duration
        self.transform = transform
        
        # Filter to target tags if specified
        if target_tags:
            self.target_tags = [t for t in target_tags if t in annotations_df.columns]
        else:
            self.target_tags = [col for col in annotations_df.columns 
                               if col not in ['clip_id', 'mp3_path']]
        
        self.n_samples = int(duration * sr)
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]
        
        # Load audio
        audio_path = self.audio_dir / row.get('mp3_path', f"{row['clip_id']}.mp3")
        
        try:
            waveform, sr = torchaudio.load(str(audio_path))
            
            # Resample if needed
            if sr != self.sr:
                resampler = torchaudio.transforms.Resample(sr, self.sr)
                waveform = resampler(waveform)
            
            # Convert to mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            
            # Pad or trim to fixed length
            if waveform.shape[1] < self.n_samples:
                padding = self.n_samples - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, padding))
            else:
                waveform = waveform[:, :self.n_samples]
            
        except Exception as e:
            print(f"Error loading {audio_path}: {e}")
            waveform = torch.zeros(1, self.n_samples)
        
        # Get labels
        labels = torch.tensor(row[self.target_tags].values.astype(float), dtype=torch.float32)
        
        if self.transform:
            waveform = self.transform(waveform)
        
        return waveform.squeeze(0), labels

print("Dataset class defined. Ready for training!")

## 5. Save Preprocessing Configuration

In [ ]:
# Save configuration for model training
config = {
    'dataset': 'MagnaTagATune',
    'sample_rate': 22050,
    'duration': 29.0,
    'n_mels': 128,
    'n_mfcc': 20,
    'hop_length': 512,
    'n_fft': 2048,
    'amapiano_target_bpm_range': (112, 118),
    'target_tags': amapiano_relevant_tags
}

import json
with open('preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("Configuration saved!")
print(json.dumps(config, indent=2))